# Drawdown in a water-table aquifer, and where superposition stops being exact

[`mf6-adj-theis`](mf6-adj-theis.ipynb) rebuilds drawdown by superposing adjoint
sensitivities, and gets it exactly right — to about 1e-11 ft — on a confined
aquifer. It ends by naming the assumption that makes that work: the aquifer has
to respond in proportion, so that doubling the pumping doubles the drawdown.

This notebook breaks that assumption on purpose. The aquifer here is the
three-layer system from [`flopy-intro-gwf-only-a`](flopy-intro-gwf-only-a.ipynb)
— a **water-table** layer, a thin **confining unit**, and a **confined aquifer**
beneath it — refined, run through eight years, and pumped by two wells: a shallow one
screwed into the water table itself and a deeper one in the confined aquifer. Because the top layer is unconfined, its transmissivity is
its conductivity times the *saturated thickness*, and the saturated thickness
falls as the water table does. Pump harder and the aquifer gets worse at
transmitting water, so the response is no longer proportional.

By the end of this notebook you will be able to:

- rebuild the drawdown in both the water table and the confined aquifer from one
  set of adjoint sensitivities,
- say how far off that rebuild is, and why it is off in the direction it is,
- predict the drawdown for any combination of pumping rates without running the
  model again, and
- read the depth to water those rates would leave behind.


## The idea

That "how much would the output change" is a **derivative**, and a well's
pumping rate is one of the parameters it can be taken with respect to. Written
for the head $h$ at an observation well and the rate $Q_i$ of well $i$,

$$\frac{\partial h}{\partial Q_i},$$

it is the **unit response**: the head change that well causes per unit of
pumping. Multiply it by the rate the well actually pumps and add up the wells,
and you have the drawdown:

$$s(t) = -\sum_i \sum_{\tau} \frac{\partial h(t)}{\partial Q_i(\tau)}\, Q_i(\tau).$$

The inner sum over $\tau$ runs over the stress periods, because a rate applied
in an earlier period still affects the head later.

Adding the wells up like this only works if the aquifer responds in proportion:
pump twice as hard and the drawdown doubles, and two wells together draw the
water level down by the sum of what each would do alone. Groundwater flow behaves
that way when the aquifer stays fully saturated and the boundaries do not move,
which is what **linear** means here. Where the sum fails to reproduce the
simulated drawdown, the difference measures how far the model departs from it.


Import the packages this notebook uses, and locate the MODFLOW 6 executable and
shared library.


In [ ]:
import pathlib as pl

import flopy
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from IPython.display import display
from mf6_notebook_helpers import find_mf6_libraries

lib_name, mf6_exe = find_mf6_libraries()

## The aquifer

The grid covers the same 10,000 by 10,500 ft domain as the introductory model,
at twice the resolution so the maps later have something to show. Layer 1
is the water table, layer 2 is a confining unit two orders of magnitude tighter
than the aquifers around it, and layer 3 is the confined aquifer. One well is screened in
each aquifer, so you can watch a water-table well and a confined well side by
side. A river runs down the eastern edge in layer 1 and recharge falls on
the whole surface.

The first stress period is steady state and establishes the starting condition.
Eight annual periods follow, with the shallow well starting in the second and the
deep well in the fifth.


In [ ]:
print(
    f"grid: {adjh.WT_NROW} rows x {adjh.WT_NCOL} columns of "
    f"{adjh.WT_DELR:.0f} by {adjh.WT_DELC:.0f} ft"
)
print(
    f"layers: water table (icelltype {adjh.WT_ICELLTYPE[0]}), "
    f"confining unit k = {adjh.WT_K[1]}, confined aquifer k = {adjh.WT_K[2]} ft/d"
)
print(f"land surface {adjh.WT_TOP:.0f} ft, layer bottoms {adjh.WT_BOTM}")
print()
for name, (k, i, j, q, start) in adjh.WT_WELLS.items():
    print(
        f"  {name} well: {q:,.0f} ft3/d from period {start + 1}, "
        f"layer {k + 1} row {i + 1} column {j + 1}"
    )

Run the model twice. The **baseline** run has both wells present but pumping
nothing, and is the state the drawdown is measured from. The second run pumps
them at their full rates, and the difference between the two is the simulated
drawdown the superposition has to reproduce.


In [ ]:
baseline_ws = pl.Path("models/adj-wt-baseline")
pumping_ws = pl.Path("models/adj-wt-pumping")

no_pumping = adjh.wt_rates({name: 0.0 for name in adjh.WT_WELLS})
for ws, rates in ((baseline_ws, no_pumping), (pumping_ws, None)):
    sim = adjh.wt_simulation(ws, mf6_exe, rates=rates)
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally"

head_baseline = adjh.wt_period_heads(baseline_ws)
head_pumping = adjh.wt_period_heads(pumping_ws)
simulated = head_baseline - head_pumping

saturated = head_baseline[-1, 0] - adjh.WT_BOTM[0]
print(
    f"saturated thickness of the water table layer: "
    f"{saturated.min():.0f} to {saturated.max():.0f} ft"
)
print(
    f"simulated drawdown at the end - water table: "
    f"{simulated[-1, 0].max():.2f} ft, confined: {simulated[-1, 2].max():.2f} ft"
)

**What to look for.** The water table sits 100 to 122 ft above the bottom of
layer 1, and the pumping pulls it down by up to about 4.5 ft. That is a few per
cent of the saturated thickness — enough to matter, but not enough to dry
anything out. Keep that ratio in mind: it is what sets the size of the error
below.


## One backward solve per well

The measures go at the *wells*, not at the observation points, because a well
and an observation point can be exchanged — the drawdown at B from pumping at A
equals the drawdown at A from the same pumping at B, the symmetry called
**reciprocity**. One backward solve per well per period therefore returns that
well's drawdown response at every cell in the model, in every layer, at once.

Two wells over nine stress periods is eighteen measures, and they are solved
about the **baseline** run: the derivatives are taken where the wells pump
nothing, which is the state the drawdown is measured from.


In [ ]:
measures = {}
for name, (k, i, j, _, _) in adjh.WT_WELLS.items():
    for kper in range(adjh.WT_NPER):
        kstp = 0 if kper == 0 else adjh.WT_NSTP - 1
        measures[f"{name}{kper:02d}"] = [(kper, kstp, k, i, j, "head")]

adj_file = adjh.write_adj_file(baseline_ws, "wt.adj", measures)
adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(baseline_ws),
)
adj.solve_forward_model()
adj.solve_adjoint()
adj.finalize()

kernels = adjh.wt_kernels(baseline_ws)
print(f"solved {len(measures)} performance measures")

## Rebuild the drawdown in both layers

Superpose as before. The same stored sensitivities give the drawdown in the
water table and in the confined aquifer at the same map location, because the
response was returned for every cell.


In [ ]:
obs_cells = {}
for name, (i, j) in adjh.WT_OBS.items():
    obs_cells[f"{name} (water table)"] = (0, i, j)
    obs_cells[f"{name} (confined)"] = (2, i, j)

predicted = adjh.wt_superpose(kernels, adjh.wt_rates(), obs_cells)

years = np.arange(adjh.WT_NPER)
print(
    f"{'observation':>44}{'simulated':>11}{'superposed':>12}{'difference':>12}{'%':>7}"
)
for name, cell in obs_cells.items():
    sim_v, pred_v = simulated[-1][cell], predicted[name][-1]
    print(
        f"{name:>44}{sim_v:11.3f}{pred_v:12.3f}{pred_v - sim_v:12.3f}"
        f"{100 * (pred_v - sim_v) / sim_v:7.1f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, constrained_layout=True)
for ax, layer, title in (
    (axes[0], 0, "water table (layer 1)"),
    (axes[1], 2, "confined aquifer (layer 3)"),
):
    for n, (name, (i, j)) in enumerate(adjh.WT_OBS.items()):
        colour = f"C{n}"
        ax.plot(
            years,
            simulated[:, layer, i, j],
            "-",
            color=colour,
            label=f"{name}, simulated",
        )
        ax.plot(
            years,
            predicted[f"{name} ({'water table' if layer == 0 else 'confined'})"],
            "--",
            color=colour,
            lw=1.2,
            label=f"{name}, superposed",
        )
    ax.set_title(title)
    ax.set_xlabel("stress period")
    ax.invert_yaxis()
axes[0].set_ylabel("drawdown (ft)")
axes[0].legend(fontsize=7, ncols=2)
plt.show()

**What to look for.** The superposition gets the shape of both curves right —
when each well starts, how fast the drawdown builds, how the two wells add
together — but it does not land on the simulated drawdown the way it did on the
confined aquifer of the Theis notebook. It runs between about 2.5 and 4.5 per
cent high, always high and never low, and the gap widens as the drawdown grows.

**That difference is the finding, not a defect.** The sensitivities were taken
where the wells pump nothing, so they describe an aquifer whose water table is
still full. Pumping thins the water-table layer, its transmissivity falls with
its saturated thickness, and the real aquifer arrives at a slightly different
answer than a proportional one would. The prediction is high because it keeps
crediting the aquifer with the transmissivity it started with.

The pattern in the numbers says the same thing. The confined readings sit in a
narrow band near 3 per cent, while the water-table readings are the scattered
ones — best where the drawdown is smallest and worst midway between the wells,
where both cones overlap and the layer is thinned by each of them. The
confined aquifer only feels the nonlinearity second-hand, through the confining
unit, so it is the steadier of the two. Linearising about the pumping run
instead makes it worse, not better, because the drawdown is being measured from
the unpumped state.

So superposition is not exact here, and cannot be. What it still is, is close
enough to be useful — and free.


## Try any pumping rates you like

The sensitivities do not depend on the rates they were computed with, so the
same eighteen solves price any combination of them. Set a multiplier on each
well and the drawdown maps below are rebuilt from the stored sensitivities
alone, with no model run.

The bottom row shows the **depth to water** — land surface minus the water-table
head — which is what a driller would measure, and which the drawdown alone does
not tell you.


In [ ]:
MULTIPLIERS = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0)
extent = (0, adjh.WT_NCOL * adjh.WT_DELR, 0, adjh.WT_NROW * adjh.WT_DELC)
base_rates = adjh.wt_rates()
base_dd = {k: adjh.wt_drawdown_map(kernels, base_rates, layer=k) for k in (0, 2)}
# Water levels are tracked at the pumping wells as well as the observation
# wells, since the pumping wells are where the water table falls furthest. All
# are read in layer 1, because depth to water means the water table wherever
# the well happens to be screened.
TRACKED = {f"{name} well": (i, j) for name, (_, i, j, _, _) in adjh.WT_WELLS.items()}
TRACKED.update(adjh.WT_OBS)
tracked_cells = {name: (0, i, j) for name, (i, j) in TRACKED.items()}
base_hist = adjh.wt_superpose(kernels, base_rates, tracked_cells)


def depth_to_water(history):
    """Depth below land surface at each tracked well, through time."""
    return {
        name: adjh.WT_TOP - (head_baseline[:, 0, i, j] - history[name])
        for name, (i, j) in TRACKED.items()
    }


def compare(shallow=1.0, deep=2.0):
    """Map the base case beside a scenario, both rebuilt from the sensitivities."""
    scen_rates = adjh.wt_rates({"shallow": shallow, "deep": deep})
    scen_dd = {k: adjh.wt_drawdown_map(kernels, scen_rates, layer=k) for k in (0, 2)}
    scen_hist = adjh.wt_superpose(kernels, scen_rates, tracked_cells)
    dtw_base, dtw_scen = depth_to_water(base_hist), depth_to_water(scen_hist)

    label = f"x{shallow:g} shallow / x{deep:g} deep"
    mosaic = [["wt base", "wt scen"], ["cf base", "cf scen"], ["dtw", "dtw"]]
    panels = {
        "wt base": (base_dd[0], "Blues", "water table drawdown, base"),
        "wt scen": (scen_dd[0], "Blues", f"water table drawdown, {label}"),
        "cf base": (base_dd[2], "Purples", "confined drawdown, base"),
        "cf scen": (scen_dd[2], "Purples", f"confined drawdown, {label}"),
    }
    # one colour scale per row, so the two columns can be read against each other
    limits = {
        "wt": (0.0, max(base_dd[0].max(), scen_dd[0].max())),
        "cf": (0.0, max(base_dd[2].max(), scen_dd[2].max())),
    }

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            mosaic, figsize=(10, 12), height_ratios=(1, 1, 0.7), layout="constrained"
        )
        for key, (data, cmap, title) in panels.items():
            row = key.split()[0]
            vmin, vmax = limits[row]
            ax = axd[key]
            im = ax.imshow(
                data,
                cmap=cmap,
                extent=extent,
                origin="upper",
                vmin=vmin,
                vmax=vmax,
                aspect="equal",
            )
            for well, (_, i, j, _, _) in adjh.WT_WELLS.items():
                ax.plot(
                    (j + 0.5) * adjh.WT_DELR,
                    (adjh.WT_NROW - i - 0.5) * adjh.WT_DELC,
                    "k^",
                    ms=8,
                )
            for _, (i, j) in adjh.WT_OBS.items():
                ax.plot(
                    (j + 0.5) * adjh.WT_DELR,
                    (adjh.WT_NROW - i - 0.5) * adjh.WT_DELC,
                    "o",
                    mfc="none",
                    mec="k",
                    ms=7,
                    mew=1.2,
                )
            ax.set_title(title, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            if key.endswith("scen"):
                fig.colorbar(
                    im, ax=[axd[f"{row} base"], ax], shrink=0.8, label="drawdown (ft)"
                )

        # depth to water through time, at the pumping wells and the observation
        # wells. Colour identifies the well; solid is the base case and dashed
        # the scenario, so the legend stays short enough to read.
        ax = axd["dtw"]
        years = np.arange(adjh.WT_NPER)
        for n, name in enumerate(TRACKED):
            ax.plot(years, dtw_base[name], "-", color=f"C{n}", label=name)
            ax.plot(years, dtw_scen[name], "--", color=f"C{n}", lw=1.3)
        ax.plot([], [], "-", color="0.35", label="base")
        ax.plot([], [], "--", color="0.35", label=label)
        ax.set_xlabel("stress period")
        ax.set_ylabel("depth to water (ft)")
        ax.set_title("depth to water at the pumping and observation wells", fontsize=9)
        ax.invert_yaxis()
        ax.legend(fontsize=7, ncols=2, loc="upper right", framealpha=0.9)
        plt.show()

    print(f"{'observation well':>30}{'water table (ft)':>22}{'confined (ft)':>20}")
    print(f"{'':>30}{'base':>11}{'scenario':>11}{'base':>10}{'scenario':>10}")
    for name, (i, j) in adjh.WT_OBS.items():
        print(
            f"{name:>30}{base_dd[0][i, j]:11.2f}{scen_dd[0][i, j]:11.2f}"
            f"{base_dd[2][i, j]:10.2f}{scen_dd[2][i, j]:10.2f}"
        )

In [ ]:
controls = {
    "shallow": widgets.Dropdown(
        options=MULTIPLIERS, value=1.0, description="shallow well"
    ),
    "deep": widgets.Dropdown(options=MULTIPLIERS, value=2.0, description="deep well"),
}
display(
    widgets.VBox(list(controls.values())),
    widgets.interactive_output(compare, controls),
)

**What to look for.** Every change you make is priced without running MODFLOW 6
— the maps come from multiplying stored arrays. Set the shallow well to zero and
its cone disappears from the water table while the deep well's cone is
untouched, which is superposition doing exactly what it promises. Each well
leaves a mark on the other aquifer too, spread wide and shallow by the confining
unit between them.

The two aquifers make very different cones, which is why the observation wells
(open circles) sit where they do. The water table carries its water in about
5,500 ft²/d of transmissivity against the confined aquifer's 40,000, so its cone
is steep and narrow — the drawdown falls by a quarter within 500 ft of the
shallow well — and an observation well has to be close to see it. The confined
cone is broad and flat, and a well 2,000 ft from the deep well still reads most
of the drawdown.

The bottom panel is the one to take away. Drawdown is a *change*, so it says
nothing about where the water actually is; adding the baseline back gives the
depth below land surface a driller would measure. The two pumping wells and the
three observation wells are all on it, so you can see the water level fall
furthest at the pumping wells themselves and least at the observation well
furthest out. Watch each curve step down as its well starts and then flatten as
the aquifer finds a new balance, and note how much of the depth is the setting
rather than the pumping — the wells move it a few feet on a depth of sixty.


## Check one scenario against the model

The predictions above are only as good as the linearisation, so it is worth
running one of them for real. Take the scenario the controls start on — the
deep well at twice its rate — and compare.


In [ ]:
check_ws = pl.Path("models/adj-wt-check")
check_rates = adjh.wt_rates({"shallow": 1.0, "deep": 2.0})
check_sim = adjh.wt_simulation(check_ws, mf6_exe, rates=check_rates)
check_sim.write_simulation(silent=True)
success, buff = check_sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

check_dd = head_baseline - adjh.wt_period_heads(check_ws)
check_pred = adjh.wt_superpose(kernels, check_rates, obs_cells)

print(f"{'observation':>44}{'simulated':>11}{'predicted':>11}{'%':>8}")
for name, cell in obs_cells.items():
    sim_v, pred_v = check_dd[-1][cell], check_pred[name][-1]
    print(f"{name:>44}{sim_v:11.3f}{pred_v:11.3f}{100 * (pred_v - sim_v) / sim_v:8.1f}")

**What to look for.** The scenario is predicted about as well as the base case
was — a few per cent high, worse in the water table than in the confined
aquifer. Doubling a well's rate does not make the prediction noticeably worse,
because the error is dominated by the linearisation itself rather than by how
far the rates have moved from it.

That is the practical shape of the method on a water-table aquifer. Use the
sensitivities to screen scenarios, compare them against each other, and find the
few worth looking at closely — then run those few for real, because the model
and the superposition will not quite agree.


## Recap

- The adjoint sensitivity of a head to a well's rate is that well's **unit
  response** — the drawdown it causes per unit of pumping.
- **Reciprocity** puts the measure at the well rather than the observation
  point, so one backward solve per well returns the response everywhere, in
  every layer, at once.
- Superposing those responses rebuilds the drawdown in both the water table and
  the confined aquifer, and prices any set of pumping rates without another
  model run.
- On this aquifer the rebuild is **not exact**, and cannot be. The water-table
  layer's transmissivity follows its saturated thickness, so the aquifer does
  not respond in proportion to the pumping, and a superposition of derivatives
  taken at the unpumped state runs a few per cent high — between about 2.5 and
  4.5 per cent here, and always high rather than low.
- That is the difference between this notebook and
  [`mf6-adj-theis`](mf6-adj-theis.ipynb), where the aquifer is confined, the
  response is proportional, and the same superposition is exact to machine
  precision. Which of the two a real model resembles is worth knowing before
  trusting the reconstruction.
